In [39]:
import pandas as pd
import numpy as np
import warnings
from sklearn.preprocessing import LabelEncoder

warnings.filterwarnings("ignore")

In [40]:
df = pd.read_csv("ipc_cleaned.csv")

print("Raw shape:", df.shape)

print(df.head()) 

Raw shape: (27694, 12)
                geographic_unit_full_name geographic_unit_name  unit_type  \
0  Aberdare Forest, Nyeri, Central, Kenya      Aberdare Forest  fsc_admin   
1  Aberdare Forest, Nyeri, Central, Kenya      Aberdare Forest  fsc_admin   
2  Aberdare Forest, Nyeri, Central, Kenya      Aberdare Forest  fsc_admin   
3  Aberdare Forest, Nyeri, Central, Kenya      Aberdare Forest  fsc_admin   
4  Aberdare Forest, Nyeri, Central, Kenya      Aberdare Forest  fsc_admin   

             fnid classification_scale  is_allowing_for_assistance  \
0  KE2011C1480601              IPC 2.0                       False   
1  KE2011C1480601              IPC 2.0                       False   
2  KE2011C1480601              IPC 2.0                       False   
3  KE2011C1480601              IPC 2.0                       False   
4  KE2011C1480601              IPC 2.0                       False   

  projection_start projection_end  value description  \
0       2011-07-01     2011-07-31    

In [41]:
print(df.columns.tolist())

['geographic_unit_full_name', 'geographic_unit_name', 'unit_type', 'fnid', 'classification_scale', 'is_allowing_for_assistance', 'projection_start', 'projection_end', 'value', 'description', 'dataseries_name', 'reporting_date']


In [42]:
df["reporting_date"] = pd.to_datetime(df["reporting_date"])

In [43]:
df = df.sort_values("reporting_date")

In [44]:
df["panel_id"] = (
    df["geographic_unit_name"]
    + "_"
    + df["geographic_unit_full_name"]
)

In [45]:
print(df.shape)

print("Unique panels:", df["panel_id"].nunique())

panel_counts = df.groupby("panel_id").size()

print(panel_counts.describe())

(27694, 13)
Unique panels: 1218
count    1218.000000
mean       22.737274
std         6.736009
min         2.000000
25%        19.000000
50%        28.000000
75%        28.000000
max        28.000000
dtype: float64


In [47]:
sample_panel = df["panel_id"].iloc[0]

print(df[df["panel_id"] == sample_panel][
    ["reporting_date", "value"]
])

   reporting_date  value
0      2011-07-01    1.0
1      2011-10-01    1.0
2      2012-01-01    1.0
3      2012-04-01    1.0
4      2012-07-01    1.0
5      2012-10-01    1.0
6      2013-01-01    1.0
7      2013-04-01    1.0
8      2013-07-01    1.0
9      2013-10-01    1.0
10     2014-01-01    1.0
11     2014-04-01    1.0
12     2014-07-01    1.0
13     2014-10-01    1.0
14     2015-01-01    1.0
15     2015-04-01    1.0
16     2015-07-01    1.0
17     2015-10-01    1.0
18     2016-02-01    1.0


In [48]:
print(df.shape)

print("Unique panels:", df["panel_id"].nunique())

panel_counts = df.groupby("panel_id").size()

print(panel_counts.describe())

(27694, 13)
Unique panels: 1218
count    1218.000000
mean       22.737274
std         6.736009
min         2.000000
25%        19.000000
50%        28.000000
75%        28.000000
max        28.000000
dtype: float64


In [49]:
panel_counts = df.groupby("panel_id").size()

print(panel_counts.value_counts().sort_index())

2      70
10      2
19    506
28    640
Name: count, dtype: int64


In [51]:
import pandas as pd
import numpy as np

# Load your base dataset
df = pd.read_csv("ipc_cleaned.csv") # Swap with your raw data file

# 1. Always sort by space and time explicitly first
df = df.sort_values(by=['geographic_unit_name', 'reporting_date']).reset_index(drop=True)

# 2. Extract basic time integers if not already present
df['reporting_date'] = pd.to_datetime(df['reporting_date'])
df['year'] = df['reporting_date'].dt.year
df['month'] = df['reporting_date'].dt.month
df['quarter'] = df['reporting_date'].dt.quarter

# Cyclical time encodings
df['month_sin'] = np.sin(2 * np.pi * df['month'] / 12)
df['month_cos'] = np.cos(2 * np.pi * df['month'] / 12)
df['quarter_sin'] = np.sin(2 * np.pi * df['quarter'] / 4)
df['quarter_cos'] = np.cos(2 * np.pi * df['quarter'] / 4)

# 3. Create a shifted target column to prevent look-ahead bias in rolling metrics
df['value_shifted'] = df.groupby('geographic_unit_name')['value'].shift(1)

# 4. Generate time-series features safely INSIDE each geographic unit group
df['lag_1'] = df.groupby('geographic_unit_name')['value'].shift(1)
df['lag_2'] = df.groupby('geographic_unit_name')['value'].shift(2)
df['lag_3'] = df.groupby('geographic_unit_name')['value'].shift(3)
df['lag_6'] = df.groupby('geographic_unit_name')['value'].shift(6)

df['rolling_mean_3'] = df.groupby('geographic_unit_name')['value_shifted'].transform(lambda x: x.rolling(3).mean())
df['rolling_mean_6'] = df.groupby('geographic_unit_name')['value_shifted'].transform(lambda x: x.rolling(6).mean())
df['rolling_std_3']  = df.groupby('geographic_unit_name')['value_shifted'].transform(lambda x: x.rolling(3).std())
df['rolling_std_6']  = df.groupby('geographic_unit_name')['value_shifted'].transform(lambda x: x.rolling(6).std())
df['rolling_min_3']  = df.groupby('geographic_unit_name')['value_shifted'].transform(lambda x: x.rolling(3).min())
df['rolling_min_6']  = df.groupby('geographic_unit_name')['value_shifted'].transform(lambda x: x.rolling(6).min())
df['rolling_max_3']  = df.groupby('geographic_unit_name')['value_shifted'].transform(lambda x: x.rolling(3).max())
df['rolling_max_6']  = df.groupby('geographic_unit_name')['value_shifted'].transform(lambda x: x.rolling(6).max())

df['difference']     = df.groupby('geographic_unit_name')['value_shifted'].transform(lambda x: x.diff(1))
df['pct_change']     = df.groupby('geographic_unit_name')['value_shifted'].transform(lambda x: x.pct_change(1) * 100)
df['expanding_mean'] = df.groupby('geographic_unit_name')['value_shifted'].transform(lambda x: x.expanding().mean())

# Clean up helper column
df = df.drop(columns=['value_shifted'])

# 5. Drop boundary rows where lags/rolling features couldn't be calculated
df_modelling = df.dropna(subset=['lag_6', 'rolling_mean_6', 'pct_change']).copy()

# Save modeling ready data
df_modelling.to_csv("food_security_model_ready.csv", index=False)
print("✅ Feature engineering complete with zero cross-panel leakage!")


✅ Feature engineering complete with zero cross-panel leakage!


In [52]:
ml_features = [
    "lag_1",
    "lag_2",
    "lag_3",
    "lag_6",
    "rolling_mean_3",
    "rolling_mean_6",
    "rolling_std_3",
    "rolling_std_6",
    "rolling_min_3",
    "rolling_min_6",
    "rolling_max_3",
    "rolling_max_6",
    "difference",
    "pct_change",
    "expanding_mean",
    "month_sin",
    "month_cos",
    "quarter_sin",
    "quarter_cos"
]

model_df = df.dropna(subset=ml_features).reset_index(drop=True)

print(model_df.shape)

(24380, 34)


In [54]:
print("=" * 50)
print("FEATURE ENGINEERING COMPLETE")
print("=" * 50)

print(f"Original rows: {len(df)}")
print(f"Final modelling rows: {len(model_df)}")
print(f"Features created: {model_df.shape[1]}")


print("\nMissing Values")
print(model_df.isnull().sum().sum())

print("\nDataset Preview")
model_df.head()

FEATURE ENGINEERING COMPLETE
Original rows: 27694
Final modelling rows: 24380
Features created: 34

Missing Values
0

Dataset Preview


,geographic_unit_full_name,geographic_unit_name,unit_type,fnid,classification_scale,is_allowing_for_assistance,projection_start,projection_end,value,description,...,rolling_mean_6,rolling_std_3,rolling_std_6,rolling_min_3,rolling_min_6,rolling_max_3,rolling_max_6,difference,pct_change,expanding_mean
0,"Aberdare Forest, Nyeri, Central, Kenya",Aberdare Forest,fsc_admin,KE2011C1480601,IPC 2.0,False,2013-01-01,2013-01-31,1.0,Minimal,...,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0
1,"Aberdare Forest, Nyeri, Central, Kenya",Aberdare Forest,fsc_admin,KE2011C1480601,IPC 2.0,False,2013-04-01,2013-04-30,1.0,Minimal,...,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0
2,"Aberdare Forest, Nyeri, Central, Kenya",Aberdare Forest,fsc_admin,KE2011C1480601,IPC 2.0,False,2013-07-01,2013-07-31,1.0,Minimal,...,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0
3,"Aberdare Forest, Nyeri, Central, Kenya",Aberdare Forest,fsc_admin,KE2011C1480601,IPC 2.0,False,2013-10-01,2013-10-31,1.0,Minimal,...,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0
4,"Aberdare Forest, Nyeri, Central, Kenya",Aberdare Forest,fsc_admin,KE2011C1480601,IPC 2.0,False,2014-01-01,2014-01-31,1.0,Minimal,...,1.0,0.0,0.0,1.0,1.0,1.0,1.0,0.0,0.0,1.0


In [32]:
print(df.columns.tolist())


['geographic_unit_full_name', 'geographic_unit_name', 'unit_type', 'fnid', 'classification_scale', 'is_allowing_for_assistance', 'projection_start', 'projection_end', 'value', 'description', 'dataseries_name', 'reporting_date', 'panel_id', 'lag_1', 'lag_2', 'lag_3', 'lag_6', 'rolling_mean_3', 'rolling_mean_6', 'rolling_std_3', 'rolling_std_6', 'rolling_min_3', 'rolling_min_6', 'rolling_max_3', 'rolling_max_6', 'difference', 'pct_change', 'expanding_mean', 'year', 'month', 'quarter', 'month_sin', 'month_cos', 'quarter_sin', 'quarter_cos']


In [55]:
# Double-check that your rows are completely clean
print("Remaining rows for analysis:", len(df_modelling))
print("Total missing values left:", df_modelling[['lag_6', 'rolling_mean_6', 'pct_change']].isna().sum().sum())


Remaining rows for analysis: 24380
Total missing values left: 0


In [56]:
model_df.to_csv("food_security_model_ready.csv", index=False)

print("Dataset saved successfully.")

Dataset saved successfully.
